In [ ]:
import numpy as np
import pandas as pd
from BackEnd.database import get_connection

ModuleNotFoundError: No module named 'BackEnd'

In [ ]:
def load_data():
    conn = get_connection()
    sql = "SELECT * FROM onlyhotellist"
    df = pd.read_sql(sql, conn)
    conn.close()
    return df

In [2]:
data = {
    'type': [100, 500, 100, 500],
    'area': [101, 102, 103, 111],
    'name': ['서울호텔', '경기리조트', '인천호텔', '제주리조트'],
    'low_week_price': [150000, 250000, 120000, 350000],
    'grade': [4.5, 4.2, 3.8, 4.8],
    'reviewCount': [120, 80, 50, 300],
    'facility_list': ['수영장,조식,와이파이', '수영장,사우나', '주차장', '수영장,조식,바,클럽'],
    'swimming_pool': [2, 3, 0, 1],
    'star': [5, 4, 3, 5],
    'parking': [1, 1, 1, 1]
}

In [ ]:
df = load_data()
print(df.head())

NameError: name 'load_data' is not defined

In [ ]:
df['facility_count'] = df['facility_list'].apply(lambda x: len(x.split(','))if isinstance(x, str) else 0)

df = pd.get_dummies(df, columns=['area', 'type'], prefix=['area', 'type'])

id_cols = [col for col in df.columns if col not in ['low_week_price', 'low_weekend_price', 'event_price', 'name', 'facility_list']]

df_melted = pd.melt(
    df, 
    id_vars=id_cols, 
    value_vars=['low_week_price', 'low_weekend_price', 'event_price'],
    var_name='price_type', 
    value_name='target_price'
)

price_map = {'low_week_price': 0, 'low_weekend_price': 1, 'event_price': 2}
df_melted['day_feature'] = df_melted['price_type'].map(price_map)

df_melted['log_target_price'] = np.log1p(df_melted['target_price'])

final_df = df_melted.drop(columns=['price_type', 'target_price'])

print(f"전처리 전 데이터 개수: {len(df)}개")
print(f"전처리 후 데이터 개수: {len(final_df)}개 (3배 증폭 완료)")
print("\n--- 최종 데이터 컬럼 구성 ---")
print(final_df.columns.tolist())